<div dir="rtl">
<h1>گزارشی که از مدل جدا نمی‌شود</h1>
<p>درس 69 از 76 · چطور از آزمایش‌ها دانش قابل بازگشت بسازیم؟ · <code dir="ltr">62-journal</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/62-journal.html">📖 بازگشت به همین درس</a></p>
<p>یک نتیجهٔ واقعی را همراه تنظیمات مستقل و عدد قابل ذخیره ثبت کنید.</p><p>پیش‌نیاز: dict، JSON، Seed و تفاوت Tensor با عدد Python.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر گزارش همان dict تنظیمات را نگه دارد، تغییر تنظیمات آزمایش بعدی با سابقهٔ قبلی چه می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import copy
import json
import math
from dataclasses import asdict
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
config = ModelConfig(12,4,8,2,1,0.0)
model = MiniGPT(config)
x,y = torch.tensor([[1,2,3]]),torch.tensor([[2,3,4]])
measured_loss = model(x,y)[1]
settings = asdict(config)
print('actual untrained loss:',measured_loss.item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>make_record(settings, Seed, Loss) یک dict با کلیدهای settings، Seed و Loss بسازد. settings باید deepcopy شود و Loss یک float باشد. این گزارش وزن‌ها یا سند کامل آزمایش نیست؛ فقط ثبت یک مشاهده است.</p>
</div>

In [ ]:
def make_record(settings, seed, loss):
    # TODO: سابقه نباید با تنظیمات آزمایش بعدی تغییر کند
    return None

In [ ]:
def test_exercise():
    source = {'model':{'width':8},'rate':0.001}
    result = make_record(source,7,measured_loss)
    if result is None:
        return False
    assert result['seed']==7 and isinstance(result['loss'],float)
    assert result['loss']==measured_loss.item()
    source['model']['width']=16
    assert result['settings']['model']['width']==8
    assert json.loads(json.dumps(result))==result
    assert make_record({},8,2.0)=={'settings':{},'seed':8,'loss':2.0}
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: make_record')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Seed را تغییر دهید و هر نتیجه را همراه همان Seed چاپ کنید. ادعا نکنید مدل با Loss آغازین کمتر، پس از آموزش هم بهتر خواهد بود.</p>
</div>

In [ ]:
for seed in (7,8):
    torch.manual_seed(seed)
    trial = MiniGPT(config)
    print(json.dumps({'seed':seed,'initial_loss':trial(x,y)[1].item()}))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>Tensor خام در JSON ذخیره نمی‌شود؛ NaN هم مشاهدهٔ معتبر نیست. metric_number(Value) برای Tensor Scalar یا عدد Python یک float متناهی برگرداند؛ برای مقدار نامتناهی ValueError بدهد.</p>
</div>

In [ ]:
try:
    json.dumps({'loss':measured_loss})
except TypeError as error:
    print('expected serialization failure:',error)
else:
    raise AssertionError('a raw tensor is not JSON data')

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def metric_number(value):
    # TODO: فقط عدد متناهی و مستقل از Graph
    return None

In [ ]:
def test_repair():
    result = metric_number(measured_loss)
    if result is None:
        return False
    assert isinstance(result,float) and result==measured_loss.item()
    assert metric_number(2)==2.0
    for value in (float('nan'),float('inf')):
        try:
            metric_number(value)
        except ValueError:
            pass
        else:
            raise AssertionError('nonfinite measurement')
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: metric_number')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>مشاهده از MiniGPT واقعی است. در دفتر آزمایش و learning-log.md باید علاوه بر این عدد، سؤال، داده، فرمان، زمان و محدودیت نتیجه را ثبت کنید؛ این dict جای آن گزارش کامل نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>برای بازسازی این آزمایش روی رایانه‌ای دیگر، چه اطلاعاتی هنوز در record شما کم است؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/62-journal.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/62-journal.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>